# Try Qwen3-8B

Use the repository's `.venv` kernel and a GPU. Shut down the Pythia notebook's kernel to free its GPU memory before loading Qwen. Run the setup cells once, then edit and rerun the last cell for individual sentences. Weights load in BF16 (about 16 GB for weights, plus inference memory); the first run downloads the checkpoint if it is not cached.

This notebook uses Qwen's chat template with thinking disabled and prints only the generated answer.

In [1]:
import csv
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent

with (ROOT / "data/generated/corpus.csv").open(newline="", encoding="utf-8") as handle:
    rows = [row for row in csv.DictReader(handle) if row["split"] in {"test", "gen_test"}]


/u/dssc/zenocosini/AC-TPR-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "Qwen/Qwen3-8B"
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16,
).to(DEVICE).eval()


Loading weights: 100%|██████████| 399/399 [00:00<00:00, 8706.18it/s]


## Generate an answer

Set `EXAMPLE` to a row index, or replace `SENTENCE` with a pasted dataset sentence. The corresponding dataset answer is `rows[EXAMPLE]["agent"]`. Generation stops at EOS or 64 new tokens.

In [3]:
EXAMPLE = 0
SENTENCE = rows[EXAMPLE]["sentence"]  # Or paste a dataset sentence here.

messages = [{
    "role": "user",
    "content": f"{SENTENCE}\nWho is performing the action? Answer only with the noun identifying that person.",
}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))


The actor.
